In [10]:
import os        # 작업 폴더 변경용
import pathlib   # 경로를 객체로 다루기
import sys       # 모듈 검색 경로(sys.path) 조작용

here = pathlib.Path.cwd()                                   # 현재 작업 폴더
ROOT = here.parents[2] if here.name == "Wed" else here      # Wed에서 실행했으면 3단계 위가 프로젝트 루트
os.chdir(ROOT)                                              # 작업 폴더를 루트로 이동
SANDBOX = ROOT / "sandbox" / "W5" / "Wed"                   # 오늘 실습 폴더
SAMPLES = ROOT / "sandbox" / "W5" / "Mon" / "samples"       # 샘플 문서(재료) 폴더

print("프로젝트 루트  :", ROOT)       # 루트 경로 확인
print("재료 폴더      :", SAMPLES)    # 샘플 폴더 경로 확인

if str(ROOT / 'backend') not in sys.path:          # backend가 import 경로에 없으면
    sys.path.insert(0, str(ROOT / 'backend'))      # 맨 앞에 추가 → `from app...` import 가능

프로젝트 루트  : d:\hanwha-agent
재료 폴더      : d:\hanwha-agent\sandbox\W5\Mon\samples


In [11]:
# 말 한 마디를 대화 이력에 쌓는 함수
def append_turn(history:list[dict[str,str]], role:str, content:str)->None:
    history.append({'role':role, 'content':content})   # {역할, 내용} 딕셔너리를 이력 끝에 추가

chat: list[dict[str,str]] = []                                         # 빈 대화 이력
append_turn(chat, 'user', 'How is the weather today?')                 # 사용자 발화 추가
append_turn(chat, 'assistant', 'No access to the forecast database')   # 어시스턴트 발화 추가

for message in chat:                                        # 이력을 순서대로 돌면서
    print(f"{message['role']:<10}{message['content']}")     # 역할(10칸 왼쪽 정렬) + 내용 출력

print(f'대화 이력 길이:{len(chat)}개')                        # 쌓인 발화 수 출력

user      How is the weather today?
assistant No access to the forecast database
대화 이력 길이:2개


In [12]:
if not SAMPLES.is_dir():                     # 샘플 폴더가 없으면
    print('재료 폴더가 없습니다')              # 안내만 출력
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]                  # 하위 폴더까지 모든 파일
    top_files = [p for p in SAMPLES.iterdir() if p.is_file()]                   # samples 바로 아래 파일만
    sub_files = [p for p in (SAMPLES / '_formats').rglob('*') if p.is_file()]   # _formats 폴더 안 파일
    today = sorted((p.name for p in SAMPLES.glob('DOC-HR-014_*_v2.0.*')))       # 오늘 쓸 v2.0 파일 이름들(정렬)
    print(f'재료 파일 : {len(all_files)}개')                                      # 전체 파일 수
    print(f'  samples 바로 아래 : {len(top_files)}개 · _formats 안 : {len(sub_files)}개')  # 위치별 파일 수
    print('오늘 읽을 세 형식 :')
    for name in today:          # v2.0 파일들을
        print(' ', name)        # 한 줄씩 출력

재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
오늘 읽을 세 형식 :
  DOC-HR-014_국내출장_여비_규정_v2.0.docx
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx
  DOC-HR-014_국내출장_여비_규정_v2.0.pdf


In [13]:
from app.rag.local_parsers import parse_local                 # 파일 형식별 로컬 파서
TARGET = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))         # v2.0 docx 파일 하나 찾기
doc = parse_local(TARGET)                                     # 파싱 → 블록 목록을 가진 문서 객체
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}')   # 블록/표/쪽 수 요약

블록 233 · 표 7 · 쪽 1


In [14]:
for i, b in enumerate(doc.blocks[:5]):                           # 앞 5개 블록만
    head = b.text[:36] + ('…' if len(b.text) > 36 else '')       # 본문 앞 36자, 넘치면 … 붙이기
    print(f'[{i:2d}] {b.kind} | {b.locator} | {head}')           # 번호 | 종류 | 위치 | 미리보기
first_table = next((b for b in doc.blocks if b.kind == '표'))    # 종류가 '표'인 첫 블록
print()
print('첫 표 블록 : locator=' + first_table.locator)             # 표 위치(예: 표1)
print(' ', first_table.text[:60])                               # 표 내용(마크다운) 앞 60자

[ 0] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 일반
[ 1] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 국내출장 여비 규정
[ 2] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | v2.0
[ 3] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 문서번호  DOC-HR-014
[ 4] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 시행일    2025-07-01

첫 표 블록 : locator=표1
  | 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|--


In [15]:
# 두 문자열의 글자 집합 유사도(자카드) 계산
def jaccard(x: str, y: str) -> float:
    sx, sy = (set(x.replace(' ', '')), set(y.replace(' ', '')))   # 공백 빼고 글자 집합으로
    return len(sx & sy) / len(sx | sy)                            # 교집합 크기 / 합집합 크기
PAIRS = [('광역시 숙박비 상한', '잠자리 비용 한도'), ('광역시 숙박비 상한', '광역시 숙박비 상한액')]   # 비교할 문장 쌍
for left, right in PAIRS:                                                        # 쌍마다
    shared = sorted(set(left.replace(' ', '')) & set(right.replace(' ', '')))   # 겹치는 글자들
    shown = shared if len(shared) <= 3 else f'{len(shared)}개'                  # 3개 이하면 글자, 많으면 개수만
    print(f"'{left}' vs '{right}'  겹치는 글자 {shown} · 자카드 {jaccard(left, right):.3f}")   # 결과 출력

'광역시 숙박비 상한' vs '잠자리 비용 한도'  겹치는 글자 ['비', '한'] · 자카드 0.154
'광역시 숙박비 상한' vs '광역시 숙박비 상한액'  겹치는 글자 8개 · 자카드 0.889


In [16]:
from app.rag.local_parsers import parse_local           # 로컬 파서

DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))     # v2.0 docx 파일 찾기
doc = parse_local(DOCX)                                 # 블록 단위로 파싱

print('연 문서 :', DOCX.name, '· 블록', len(doc.blocks), '개')   # 파일명과 블록 수
print()

# SQL LIKE '%word%'처럼 단어가 들어간 블록 수 세기
def like(word: str) -> int:
    return sum((1 for b in doc.blocks if word in b.text))   # 포함된 블록마다 1씩 더함

for word in ['숙박비', '잠자리', '잠자리 비용']:       # 같은 뜻, 다른 표현으로 검색
    print(f"LIKE '%{word}%' → {like(word)}건")       # 글자가 정확히 일치해야만 찾음을 확인


연 문서 : DOC-HR-014_국내출장_여비_규정_v2.0.docx · 블록 233 개

LIKE '%숙박비%' → 27건
LIKE '%잠자리%' → 0건
LIKE '%잠자리 비용%' → 0건


In [17]:
import math                                    # 제곱근 계산용
# 문장을 2차원 좌표(임베딩 흉내)로 표현
POINTS = {'광역시 숙박비 상한': [0.9, 0.2], '잠자리 비용 한도': [0.86, 0.28], '출장 신청 절차': [0.1, 0.95]}

# 두 좌표 사이의 유클리드 거리
def distance(p: list[float], q: list[float]) -> float:
    return math.sqrt(sum(((a - b) ** 2 for a, b in zip(p, q))))   # 좌표별 차이 제곱합의 제곱근

BASE = '광역시 숙박비 상한'                      # 기준 문장
print('기준 :', BASE, POINTS[BASE])

for text, point in POINTS.items():             # 모든 문장에 대해
    if text == BASE:                           # 기준 문장 자신은
        continue                               # 건너뛰기
    gap = distance(POINTS[BASE], point)        # 기준과의 거리
    print(f"  {text}  {point} → 거리 {gap:.3f}   {('← 가깝다' if gap < 0.5 else '← 멀다')}")   # 0.5 미만이면 의미가 가깝다


기준 : 광역시 숙박비 상한 [0.9, 0.2]
  잠자리 비용 한도  [0.86, 0.28] → 거리 0.089   ← 가깝다
  출장 신청 절차  [0.1, 0.95] → 거리 1.097   ← 멀다


In [18]:
v2 = [0.9, 0.2]                                         # 2차원 벡터
v3 = [0.6, 0.8, 0.0]                                    # 3차원 벡터
v1024 = [round(math.sin(i), 4) for i in range(1024)]    # 1024차원 벡터(sin 값으로 채운 가짜 임베딩)


# 벡터의 크기(길이, ‖v‖) 계산
def size(v: list[float]) -> float:
    return math.sqrt(sum(x * x for x in v))             # 각 성분 제곱합의 제곱근


for label, v in [("2차원", v2), ("3차원", v3), ("1024차원", v1024)]:   # 세 벡터를 차례로
    # 문법 - v[:5] 는 앞에서 다섯 개만 잘라 낸 새 리스트다(슬라이싱)
    print(f"{label:<8} : 길이 {len(v):<4} 앞 5개 {v[:5]}")          # 이름 | 차원 수 | 앞 5개 값

print()
print(f"크기(‖v‖) : {size(v2):.4f} · {size(v3):.4f} · {size(v1024):.4f}")   # 세 벡터의 크기 비교(v3는 크기 1)

2차원      : 길이 2    앞 5개 [0.9, 0.2]
3차원      : 길이 3    앞 5개 [0.6, 0.8, 0.0]
1024차원   : 길이 1024 앞 5개 [0.0, 0.8415, 0.9093, 0.1411, -0.7568]

크기(‖v‖) : 0.9220 · 1.0000 · 22.6283


In [19]:
# 내적 
def dot(u: list[float], v: list[float]) -> float:
    return sum((a * b for a, b in zip(u, v)))

# 크기 
def norm(u: list[float]) -> float:
    return math.sqrt(dot(u, u))

# 코사인 유사도  
def cosine(u: list[float], v: list[float]) -> float:
    return dot(u, v) / (norm(u) * norm(v))

u = [0.6, 0.8, 0]
v = [0, 1, 0]

print('u =', u, '  v =', v)
print(f'  내적   : {dot(u, v):+.4f}')
print(f'  크기   : {norm(u):.4f} × {norm(v):.4f}')
print(f'  코사인 : {cosine(u, v):+.4f}')


u = [0.6, 0.8, 0]   v = [0, 1, 0]
  내적   : +0.8000
  크기   : 1.0000 × 1.0000
  코사인 : +0.8000


In [20]:
w = [-0.6, -0.8, 0]    # u 를 정확히 뒤집은 벡터
perp = [0, 0, 1]       # u 와 직각인 벡터 (perpendicular)

CASES = [
    ("u", u,    "똑같다"),
    ("v", v,    "닮았다"),
    ("⊥", perp, "아무 상관 없다"),
    ("w", w,    "정반대"),
]
for label, other, meaning in CASES:
    print(f"cos(u, {label}) = {cosine(u, other):+.4f}    {meaning}")

cos(u, u) = +1.0000    똑같다
cos(u, v) = +0.8000    닮았다
cos(u, ⊥) = +0.0000    아무 상관 없다
cos(u, w) = -1.0000    정반대


In [21]:
def unit(v: list[float]) -> list[float]:
    n = norm(v)
    return [x / n for x in v]

u = [0.6, 0.8, 0]
v = [4.0, 3.0, 0]
uh, vh = (unit(u), unit(v))
print(f'‖û‖ = {norm(uh):.1f}   ‖v̂‖ = {norm(vh):.1f}')

gap = abs(dot(uh, vh) - cosine(u, v))
print(f'dot(û, v̂) = {dot(uh, vh):.6f}   cosine(u, v) = {cosine(u, v):.6f}   차이 {gap:.1f}')
print('차이가 1e-12 보다 작은가 :', gap < 1e-12)

‖û‖ = 1.0   ‖v̂‖ = 1.0
dot(û, v̂) = 0.960000   cosine(u, v) = 0.960000   차이 0.0
차이가 1e-12 보다 작은가 : True


In [22]:
BYTES_PER_NUMBER = 4

def mb(dim: int, n_chunks: int) -> float:
    return dim * n_chunks * BYTES_PER_NUMBER / 1024 / 1024

CASES = [(1024, 70, ''), (1024, 8000, ''), (4096, 70, ''), (4096, 8000, '     ← 4배')]
for dim, n, note in CASES:
    print(f'{dim}차원 × {n:>4}청크 × 4바이트 = {mb(dim, n):>6.2f} MB{note}')
print()
print(f'4096차원은 1024차원의 {mb(4096, 8000) / mb(1024, 8000):.0f}배다')


1024차원 ×   70청크 × 4바이트 =   0.27 MB
1024차원 × 8000청크 × 4바이트 =  31.25 MB
4096차원 ×   70청크 × 4바이트 =   1.09 MB
4096차원 × 8000청크 × 4바이트 = 125.00 MB     ← 4배

4096차원은 1024차원의 4배다


In [ ]:
MODELS = ROOT.parent / 'models'      # download_models.py가 받는 자리: hanwha-agent 바로 옆 models/
KURE = MODELS / 'KURE-v1'            # 한국어 특화 임베딩 모델
BGE = MODELS / 'bge-m3'              # 다국어 임베딩 모델
MODELS_READY = KURE.is_dir() and BGE.is_dir()   # 두 폴더가 다 있어야 True
print('KURE-v1 :', KURE)
print('bge-m3  :', BGE)
print('MODELS_READY :', MODELS_READY)

In [23]:
def show_folder(title, folder):

    files = sorted((p for p in folder.iterdir() if p.is_file()), key=lambda p: p.stat().st_size, reverse=True)
    total = sum((p.stat().st_size for p in folder.rglob('*') if p.is_file()))
    print(f'{title}  (약 {total / 1000000000.0:.2f} GB)')
    for p in files[:5]:
        print(f'  {p.name:<24} {p.stat().st_size / 1000000.0:>8.1f} MB')
if MODELS_READY:
    show_folder('KURE-v1', KURE)
    show_folder('bge-m3 ', BGE)
else:
    print('모델 폴더가 없어 건너뜁니다. 오늘 진도는 그대로 나갑니다.')


NameError: name 'MODELS_READY' is not defined

In [24]:
PAIRS = [
    ("광역시 숙박비 상한", "잠자리 비용 한도", "같은 뜻"),
    ("출장 전에 신청서를 낸다", "출장은 사전 신청이 원칙이다", "같은 뜻"),
    ("법인카드로 결제한다", "법인카드 사용 지침을 따른다", "가까움"),
    ("일비는 하루 단위로 준다", "숙박비는 1박 단위로 준다", "가까움"),
    ("출장 신청서를 낸다", "재택근무를 신청한다", "다름"),
    ("숙박비 상한액", "정보보안 지침 위반", "남남"),
]

CLOSE = {"같은 뜻", "가까움"}

near = sum(1 for a, b, label in PAIRS if label in CLOSE)
print(f"문장쌍 {len(PAIRS)}개 (가까운 쌍 {near} · 먼 쌍 {len(PAIRS) - near})")

for i, (a, b, label) in enumerate(PAIRS, start=1):
    print(f" {i} [{label}] {a}  ↔  {b}")

문장쌍 6개 (가까운 쌍 4 · 먼 쌍 2)
 1 [같은 뜻] 광역시 숙박비 상한  ↔  잠자리 비용 한도
 2 [같은 뜻] 출장 전에 신청서를 낸다  ↔  출장은 사전 신청이 원칙이다
 3 [가까움] 법인카드로 결제한다  ↔  법인카드 사용 지침을 따른다
 4 [가까움] 일비는 하루 단위로 준다  ↔  숙박비는 1박 단위로 준다
 5 [다름] 출장 신청서를 낸다  ↔  재택근무를 신청한다
 6 [남남] 숙박비 상한액  ↔  정보보안 지침 위반


In [25]:
def jaccard(x, y):
    sx, sy = (set(x.replace(' ', '')), set(y.replace(' ', '')))
    return len(sx & sy) / len(sx | sy)
scores = {True: [], False: []}
for i, (a, b, label) in enumerate(PAIRS, start=1):
    value = jaccard(a, b)
    scores[label in CLOSE].append(value)
    print(f' {i} [{label}] 자카드 {value:.3f}')
print()
print(f'가까운 쌍 평균 : {sum(scores[True]) / len(scores[True]):.3f}')
print(f'먼 쌍 평균     : {sum(scores[False]) / len(scores[False]):.3f}')

 1 [같은 뜻] 자카드 0.154
 2 [같은 뜻] 자카드 0.400
 3 [가까움] 자카드 0.312
 4 [가까움] 자카드 0.538
 5 [다름] 자카드 0.308
 6 [남남] 자카드 0.000

가까운 쌍 평균 : 0.351
먼 쌍 평균     : 0.154
